In [1]:
import pandas as pd
import sqlite3
import json

In [3]:
movies_df = pd.read_csv("../data/raw/tmdb_5000_movies.csv")
movies_df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [5]:
credits_df = pd.read_csv("../data/raw/tmdb_5000_credits.csv")
credits_df.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [7]:
# Create connection to SQLite database
conn = sqlite3.connect("../data/cineanalytica.db")
cursor = conn.cursor()

print("Database created successfully!")

Database created successfully!


In [11]:
cursor.executescript("""
DROP TABLE IF EXISTS movies;

CREATE TABLE movies (
    movie_id INTEGER PRIMARY KEY,
    title TEXT,
    original_title TEXT,
    original_language TEXT,
    release_date TEXT,
    runtime REAL,
    budget REAL,
    revenue REAL,
    popularity REAL,
    vote_average REAL,
    vote_count REAL,
    overview TEXT
);
""")

conn.commit()
print("✅ movies table created")

✅ movies table created


In [13]:
# Insert data into movies table

for _, row in movies_df.iterrows():
    cursor.execute("""
        INSERT INTO movies (
            movie_id, title, original_title, original_language,
            release_date, runtime, budget, revenue,
            popularity, vote_average, vote_count, overview
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        row["id"],
        row["title"],
        row["original_title"],
        row["original_language"],
        row["release_date"],
        row["runtime"],
        row["budget"],
        row["revenue"],
        row["popularity"],
        row["vote_average"],
        row["vote_count"],
        row["overview"]
    ))

conn.commit()

print("✅ Movie data inserted successfully!")

✅ Movie data inserted successfully!


In [15]:
cursor.execute("SELECT COUNT(*) FROM movies;")
cursor.fetchone()

(4803,)

In [17]:
cursor.executescript("""
DROP TABLE IF EXISTS actors;

CREATE TABLE actors (
    actor_id INTEGER PRIMARY KEY,
    actor_name TEXT
);
""")

conn.commit()
print("✅ actors table created")

✅ actors table created


In [19]:
cursor.executescript("""
DROP TABLE IF EXISTS movie_cast;

CREATE TABLE movie_cast (
    movie_id INTEGER,
    actor_id INTEGER,
    character_name TEXT,
    cast_order INTEGER,
    PRIMARY KEY (movie_id, actor_id),
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id),
    FOREIGN KEY (actor_id) REFERENCES actors(actor_id)
);
""")

conn.commit()
print("✅ movie_cast table created")

✅ movie_cast table created


In [21]:
# Insert cast data into actors and movie_cast

actor_seen = set()  # to avoid inserting same actor twice

for _, row in credits_df.iterrows():
    movie_id = int(row["movie_id"])
    cast_list = json.loads(row["cast"])  # convert string -> list of dicts
    
    for c in cast_list:
        actor_id = int(c["id"])
        actor_name = c["name"]
        character_name = c.get("character", "")
        cast_order = c.get("order", None)

        # Insert into actors table only once
        if actor_id not in actor_seen:
            cursor.execute(
                "INSERT OR IGNORE INTO actors (actor_id, actor_name) VALUES (?, ?)",
                (actor_id, actor_name)
            )
            actor_seen.add(actor_id)

        # Insert into movie_cast table
        cursor.execute(
            "INSERT OR IGNORE INTO movie_cast (movie_id, actor_id, character_name, cast_order) VALUES (?, ?, ?, ?)",
            (movie_id, actor_id, character_name, cast_order)
        )

conn.commit()
print("✅ Cast data inserted into actors + movie_cast!")

✅ Cast data inserted into actors + movie_cast!


In [23]:
cursor.execute("SELECT COUNT(*) FROM actors;")
actors_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM movie_cast;")
cast_count = cursor.fetchone()[0]

actors_count, cast_count

(54588, 106084)

In [25]:
cursor.executescript("""
DROP TABLE IF EXISTS directors;

CREATE TABLE directors (
    director_id INTEGER PRIMARY KEY,
    director_name TEXT
);
""")

conn.commit()
print("✅ directors table created")

✅ directors table created


In [27]:
cursor.executescript("""
DROP TABLE IF EXISTS movie_directors;

CREATE TABLE movie_directors (
    movie_id INTEGER,
    director_id INTEGER,
    PRIMARY KEY (movie_id, director_id),
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id),
    FOREIGN KEY (director_id) REFERENCES directors(director_id)
);
""")

conn.commit()
print("✅ movie_directors table created")

✅ movie_directors table created


In [29]:
director_seen = set()

for _, row in credits_df.iterrows():
    movie_id = int(row["movie_id"])
    crew_list = json.loads(row["crew"])  # string -> list of dicts
    
    for c in crew_list:
        if c.get("job") == "Director":
            director_id = int(c["id"])
            director_name = c["name"]

            # Insert director only once
            if director_id not in director_seen:
                cursor.execute(
                    "INSERT OR IGNORE INTO directors (director_id, director_name) VALUES (?, ?)",
                    (director_id, director_name)
                )
                director_seen.add(director_id)

            # Link movie to director
            cursor.execute(
                "INSERT OR IGNORE INTO movie_directors (movie_id, director_id) VALUES (?, ?)",
                (movie_id, director_id)
            )

conn.commit()
print("✅ Directors inserted!")

✅ Directors inserted!


In [31]:
cursor.execute("SELECT COUNT(*) FROM directors;")
directors_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM movie_directors;")
movie_directors_count = cursor.fetchone()[0]

directors_count, movie_directors_count

(2578, 5166)

In [33]:
query = """
SELECT m.title, d.director_name
FROM movies m
JOIN movie_directors md ON m.movie_id = md.movie_id
JOIN directors d ON md.director_id = d.director_id
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,title,director_name
0,Avatar,James Cameron
1,Pirates of the Caribbean: At World's End,Gore Verbinski
2,Spectre,Sam Mendes
3,The Dark Knight Rises,Christopher Nolan
4,John Carter,Andrew Stanton
5,Spider-Man 3,Sam Raimi
6,Tangled,Byron Howard
7,Tangled,Nathan Greno
8,Avengers: Age of Ultron,Joss Whedon
9,Harry Potter and the Half-Blood Prince,David Yates


In [1]:
import sys
print(sys.version)

3.12.12 (main, Oct  9 2025, 11:07:00) [Clang 17.0.0 (clang-1700.6.3.2)]


In [3]:
import surprise
import shap
print("All good 🚀")

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


All good 🚀
